<div style="background:linear-gradient(135deg,#1b1448 0%,#3b1f5e 55%,#7c2d12 100%);border-radius:18px;padding:34px 30px;color:#fff;font-family:Inter,Segoe UI,sans-serif">
  <div style="font-size:12px;letter-spacing:3px;color:#fed7aa;font-weight:700;text-transform:uppercase">Chapter 4 · Your first model call</div>
  <div style="font-size:34px;font-weight:900;line-height:1.1;margin:10px 0 6px">Your first model call</div>
  <div style="font-size:15px;color:#ffe4d6;max-width:740px;line-height:1.6">The Messages API through Anthropic's SDK: a request's fields, the roles in a conversation, the system prompt, and the replay switch every notebook in the book uses.</div>
  <div style="font-size:13px;color:#fed7aa;margin-top:14px">Building AI with AI: A Visual Handbook · John Fisher</div>
</div>

Every model call in this book goes through one endpoint, the Messages API, and one library, Anthropic's Python SDK. This notebook makes the smallest request and reads what comes back, prints the JSON the SDK sends, then varies the two inputs that shape an answer most, the conversation in `messages` and the system prompt. It ends with the replay helper behind every notebook's Setup and the command-line client Claude Code generated for the chapter, run against the same recorded answers. All of it runs on Claude Sonnet 5.5.

## Setup

This cell installs Anthropic's Python library and builds the client every call below goes through. It replays recorded answers unless you set `LIVE = True`.

In [1]:
# Set LIVE = True to send these requests to the Anthropic API with your own key, saved in
# Colab's secrets panel as ANTHROPIC_API_KEY. Left False, the notebook replays the answers
# recorded for the book, with no key and no cost.
LIVE = False
%pip install -q --disable-pip-version-check anthropic==2.7.0
import importlib, pathlib, sys, urllib.request
if "google.colab" in sys.modules:
    BASE = "https://raw.githubusercontent.com/johnfisher-ai/ai-with-ai/main/chapters/notebooks/"
    pathlib.Path("recordings").mkdir(exist_ok=True)
    for name in ["book_client.py", "recordings/your-first-model-call.json"]:
        urllib.request.urlretrieve(BASE + name, name)
import book_client
importlib.reload(book_client)
client = book_client.make_client("your-first-model-call", mode="live" if LIVE else None)

Note: you may need to restart the kernel to use updated packages.


Replaying 17 recorded responses from claude-sonnet-5-5, recorded 2026-11-12. Set LIVE = True in Setup to call the API yourself.


## The client from the chapter's Claude Code build

The notebook builds its requests with `ask.py`, a command-line client Claude Code generated in the chapter's build from a spec and 23 tests. The file lives in the book's repository, in `builds/your-first-model-call/generated/`, next to the spec, the tests and a transcript of what Claude Code did. This cell loads it and prints its source, so `build_request`, `answer_text`, `format_usage`, `chat` and `main` are on the page before the notebook calls them. In Colab the cell downloads the file next to the notebook, and the Files panel on the left shows it.

In [2]:
# ask.py is the command-line client Claude Code generated in this chapter's build, from the spec and tests
# in builds/your-first-model-call/. Colab fetches it from the book's repository.
BUILD = "your-first-model-call"
if "google.colab" in sys.modules:
    urllib.request.urlretrieve("https://raw.githubusercontent.com/johnfisher-ai/ai-with-ai/main/builds/"
                               f"{BUILD}/generated/ask.py", "ask.py")
else:
    sys.path.insert(0, str(pathlib.Path("../../builds", BUILD, "generated").resolve()))
import ask
ask = importlib.reload(ask)
from ask import answer_text, build_request, chat, format_usage, main

# The whole module, so you can read every function before the notebook uses it
import inspect
print(inspect.getsource(ask))

import argparse
import sys

DEFAULT_MODEL = "claude-sonnet-5-5"
DEFAULT_MAX_TOKENS = 1024


def build_request(prompt, history=None, system=None, model=DEFAULT_MODEL,
                  max_tokens=DEFAULT_MAX_TOKENS):
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("prompt is empty")
    if max_tokens < 1:
        raise ValueError("max_tokens must be at least 1")
    history = list(history) if history else []
    for i, m in enumerate(history):
        expected = "user" if i % 2 == 0 else "assistant"
        role = m.get("role") if isinstance(m, dict) else None
        if role != expected:
            raise ValueError("invalid history")
    if history and history[-1]["role"] != "assistant":
        raise ValueError("history must end with an assistant message")
    request = {"model": model, "max_tokens": max_tokens,
               "messages": history + [{"role": "user", "content": prompt}]}
    if system:
        request["system"] = system
    return re

## The smallest request

`messages.create` needs three fields. `model` names the model, `max_tokens` caps how many tokens the reply may use, and `messages` holds the conversation, here a single user message. The response is a `Message` object, and the cell prints the fields an application reads first.

In [3]:
MODEL = "claude-sonnet-5-5"

# The smallest request: a model, a cap on the reply's length, and one user message
reply = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=[{"role": "user", "content": "In one sentence, what is a context window?"}],
)

print("type:", reply.type, "  role:", reply.role, "  model:", reply.model)
print("stop_reason:", reply.stop_reason)
print("content blocks:", [block.type for block in reply.content])
print("usage:", format_usage(reply.usage))
print()
print(answer_text(reply))

type: message   role: assistant   model: claude-sonnet-5-5
stop_reason: end_turn
content blocks: ['text']
usage: 20 tokens in, 60 out

A context window is the maximum amount of text (measured in tokens) that a language model can consider at once, including both your input and its own output, so anything beyond that limit is effectively "forgotten."


The reply is a message with the role `assistant`, from the model the request named. It stopped with `end_turn`, which means the model finished on its own rather than at `max_tokens`. Its content is a list of blocks, here one `text` block, and `answer_text` joins the text blocks so the code never assumes the answer sits in the first block. The 20 input tokens and 60 output tokens are what the call was billed for, and Chapter 5, Reading a response, takes the rest of these fields apart.

## What the SDK sends

The SDK turns the call into one HTTPS request to `/v1/messages`. The recording keeps the request's method, path and JSON body exactly as they were sent, which makes it the simplest place to see the wire format.

In [4]:
import json

# The recording keeps each request exactly as the SDK sent it, so it shows the JSON on the wire
recording = json.loads(pathlib.Path("recordings/your-first-model-call.json").read_text())
first = recording["exchanges"][0]
print(first["request"]["method"], first["request"]["path"])
print(json.dumps(first["request"]["body"], indent=2))

POST /v1/messages
{
  "max_tokens": 1024,
  "messages": [
    {
      "role": "user",
      "content": "In one sentence, what is a context window?"
    }
  ],
  "model": "claude-sonnet-5-5"
}


The body is the three fields of the call and nothing else, so a request in any other language, or with `curl`, carries the same JSON. The SDK adds the headers, among them the API key and the API version, and neither is ever written to a recording.

## The model sees only the request

The API keeps no state between calls. Whatever the model should know about the conversation has to be in `messages`, in turns that alternate between `user` and `assistant`, starting with `user`. The cell asks the same question twice, once alone and once after two earlier turns written by the application.

In [5]:
SYSTEM = "Reply in one sentence."
QUESTION = "What's my order number?"

# Without history, the model sees only this one message
alone = client.messages.create(**build_request(QUESTION, system=SYSTEM))
print("without history:", answer_text(alone))

# With history, the earlier turns travel in the same request. The assistant turn is written by the
# application, and the model reads it as its own earlier reply.
HISTORY = [
    {"role": "user", "content": "My order 48213 arrived with a cracked screen."},
    {"role": "assistant", "content": "I'm sorry it arrived damaged. I can help you arrange a replacement."},
]
remembered = client.messages.create(**build_request(QUESTION, history=HISTORY, system=SYSTEM))
print("with history:   ", answer_text(remembered))

without history: I don't have access to your order information, so please check your confirmation email, your account's order history, or contact the seller's support team to find your order number.
with history:    Your order number is 48213.


Asked alone, the model said it had no access to the order information. With the two earlier turns in the request it answered "Your order number is 48213." The assistant turn in that history was written by the application, and the model read it as its own earlier reply, so the history a program sends is the only memory the model has.

## Examples written as earlier turns

The same mechanism steers the shape of an answer. Two worked examples, each a user turn followed by the assistant turn the application wants, go in front of the real question. The cell classifies three reviews with and without them.

In [6]:
REVIEWS = [
    "The headphones sound great, but the case cracked within a week.",
    "Arrived two days early and works perfectly.",
    "Customer support never answered my emails.",
]
INSTRUCTION = "Classify the sentiment of this review as positive, negative or mixed.\n\nReview: "

# Two worked examples, written as earlier turns, show the format the answer should take
EXAMPLES = [
    {"role": "user", "content": INSTRUCTION + "Battery life is excellent."},
    {"role": "assistant", "content": "positive"},
    {"role": "user", "content": INSTRUCTION + "The charger stopped working, though the refund was quick."},
    {"role": "assistant", "content": "mixed"},
]

for review in REVIEWS:
    zero = client.messages.create(**build_request(INSTRUCTION + review))
    few = client.messages.create(**build_request(INSTRUCTION + review, history=EXAMPLES))
    print(review)
    print(f"  no examples ({zero.usage.output_tokens:>3} tokens out): {answer_text(zero)!r}")
    print(f"  two examples ({few.usage.output_tokens:>2} tokens out): {answer_text(few)!r}\n")

The headphones sound great, but the case cracked within a week.
  no examples ( 63 tokens out): '**Mixed**\n\nThe review has one positive point (the headphones sound great) and one negative point (the case cracked within a week). Neither outweighs the other, so the overall sentiment is mixed.'
  two examples ( 5 tokens out): 'mixed'

Arrived two days early and works perfectly.
  no examples ( 68 tokens out): '**Positive**\n\nThe reviewer praises both the delivery ("arrived two days early") and the product\'s performance ("works perfectly"). There are no complaints or reservations, so the sentiment is clearly positive.'
  two examples ( 4 tokens out): 'positive'

Customer support never answered my emails.
  no examples ( 58 tokens out): "**Negative**\n\nThe reviewer complains that customer support never responded to their emails. That's a clear expression of dissatisfaction, and the review includes nothing positive to balance it."
  two examples ( 4 tokens out): 'negative'



Without examples, each answer was a bold label and a paragraph of reasons, 58 to 68 output tokens. With the two examples in front, the model answered `mixed`, `positive` and `negative`, in 4 or 5 output tokens each, a reply code can compare with a string. The labels agreed in both versions, and only the format changed. Chapter 17, Examples in the prompt, chooses examples with more care.

## The system prompt sets the frame

The `system` field sits outside the conversation and tells the model how to behave on every turn. The cell asks one question under three system prompts and prints each answer with its usage.

In [7]:
QUESTION = "My package is late. What should I do?"
SYSTEM_PROMPTS = {
    "no system prompt": None,
    "one sentence": "Answer in one sentence.",
    "JSON for an app": ("You are the support assistant for an online electronics store. Reply with a JSON object "
                        "that has the keys action and reply, and nothing else."),
}

answers = {}
for name, system in SYSTEM_PROMPTS.items():
    message = client.messages.create(**build_request(QUESTION, system=system))
    answers[name] = answer_text(message)
    print(f"== {name} ({format_usage(message.usage)})")
    print(answers[name], "\n")

# The third answer is meant for code, so check that code can read it
try:
    parsed = json.loads(answers["JSON for an app"])
    print("parsed as JSON, keys:", sorted(parsed))
except json.JSONDecodeError as e:
    print("not valid JSON:", e)

== no system prompt (18 tokens in, 644 out, 21 of them thinking)
Here's a practical order of steps:

**1. Check the tracking**
- Look at the latest status and location. Delays from weather, high volume, or a missed scan are common and often resolve within a day or two.
- Confirm the delivery address on the order is correct.

**2. Look for a nearby delivery**
- Check your porch, mailbox, side door, and with neighbors.
- Ask others in your household or building. Packages are sometimes left with the front desk or office.
- Look for a delivery notice or photo confirmation.

**3. Wait a short window**
- If tracking says "delivered" but it isn't there, wait 24-48 hours. It's sometimes marked early.
- If it's just running behind, give it until the estimated delivery window passes by a day or two.

**4. Contact the seller**
- Reach out through the store or marketplace (Amazon, Etsy, eBay, etc.).
- Sellers are usually responsible for getting the package to you, and they can open a trace with th

With no system prompt, the model wrote a numbered guide of 644 output tokens. "Answer in one sentence." brought it to 58, and the JSON instruction produced an object with the keys `action` and `reply` that `json.loads` read without complaint. The JSON reply also used 136 thinking tokens, and its `action` value, `request_order_info`, is a name the model chose itself. An instruction alone doesn't guarantee that a reply parses or that its values come from an agreed list, which is what Chapter 19, Structured output, adds.

## A conversation must end with the user

Older Claude models let a request end with a partial assistant turn, which the model would continue. The cell tries it, then asks `build_request` for the same shape.

In [8]:
import anthropic

# A request that ends with an assistant turn asks the model to continue that turn
try:
    client.messages.create(model=MODEL, max_tokens=1024, messages=[
        {"role": "user", "content": "Classify the sentiment of: 'The box was crushed.'"},
        {"role": "assistant", "content": "Sentiment:"},
    ])
except anthropic.BadRequestError as e:
    print("status", e.status_code, "-", e.body["error"]["message"])

# ask.py refuses the same request before anything is sent
try:
    build_request("Classify it.", history=[{"role": "user", "content": "The box was crushed."}])
except ValueError as e:
    print("build_request:", e)

status 400 - This model does not support assistant message prefill. The conversation must end with a user message.
build_request: history must end with an assistant message


The API answered with a 400 error, "This model does not support assistant message prefill. The conversation must end with a user message.", as of November 2026. `build_request` refuses the same shape before anything is sent, because its spec requires a history to end with an assistant turn so that the new question is the last message.

## How the replay switch works

Every notebook's Setup builds its client with `book_client.make_client`. In replay mode, the default, the client's requests never leave the notebook. Each one is answered from the recording, which is found by a hash of the request's method, path, query and body. `LIVE = True` sends the same requests to the API with your own key, from Colab's secrets panel or the `ANTHROPIC_API_KEY` environment variable.

In [9]:
import os
from collections import Counter

print(f"{recording['notebook']}.json holds {len(recording['exchanges'])} exchanges")
print("models:", dict(Counter(e["model"] for e in recording["exchanges"])))
print("recorded:", sorted({e["recorded"] for e in recording["exchanges"]}), " SDK:", sorted({e["sdk"] for e in recording["exchanges"]}))

# Each exchange is found by a hash of its request, so the first request has this key
req = first["request"]
print("key of the first request:", book_client.request_key(req["method"], req["path"], req["query"], req["body"]))

# Change one character of that request and replay has nothing to answer with
MODE = "live" if LIVE else os.environ.get("BOOK_MODE", "replay")
if MODE == "replay":
    try:
        client.messages.create(model=MODEL, max_tokens=1024,
                               messages=[{"role": "user", "content": "In one sentence, what is a context window??"}])
    except anthropic.NotFoundError as e:
        print("\n" + e.body["error"]["message"])
else:
    print(f"\n{MODE} mode: the changed request would be sent to the API, so it is skipped")

your-first-model-call.json holds 17 exchanges
models: {'claude-sonnet-5-5': 17}
recorded: ['2026-11-12']  SDK: ['2.7.0']
key of the first request: be1eaf77e1906f1c

book_client: no recorded response for this request (key 23989e7e310591dd) in recordings/your-first-model-call.json. The request differs from the one the notebook recorded. Restore the cell's original code, or set LIVE = True in Setup and use your own key.


The recording holds 17 exchanges, all with Claude Sonnet 5.5, recorded on one date with SDK 2.7.0. Adding a second question mark to the first request changed its key, so replay had no answer for it and raised the SDK's `NotFoundError` with a message that says how to fix it. A notebook cell that is edited therefore fails loudly in replay mode instead of printing an answer that belongs to a different request.

## The same calls from the command line

`main` is what `python3 ask.py ...` runs. In the notebook it receives the replaying client and writes to the notebook. On your own machine it builds `anthropic.Anthropic()`, which reads your key from `ANTHROPIC_API_KEY`. The first cell asks one question with `--usage`. The second runs `--chat` with two typed lines, sending each with the conversation so far.

In [10]:
import io

# main() is what `python3 ask.py ...` runs. Here it gets the notebook's client and prints to the notebook.
main(["--usage", "--system", "Answer in one sentence.", "What does max_tokens limit?"], client=client)

`max_tokens` caps the maximum number of tokens the model can generate in its response (the output length), not the size of your input prompt, and the response is cut off if it reaches that limit.
[28 tokens in, 59 out]


0

In [11]:
# --chat reads one message per line, sends each with the conversation so far, and prints every reply
typed = io.StringIO("What's my order number?\nThanks, that's all.\n")
main(["--chat", "--usage", "--system", "You are a support assistant. Reply in one or two sentences.",
      "My order 48213 arrived with a cracked screen."], client=client, stdin=typed)

I'm sorry your order 48213 arrived with a cracked screen. Could you send a photo of the damage and the packaging, and tell me whether you'd prefer a replacement or a refund so I can get that started?
[45 tokens in, 138 out, 69 of them thinking]

Your order number is 48213, the one you mentioned earlier. Let me know if you'd like to go ahead with a replacement or refund for the cracked screen.
[193 tokens in, 52 out]

You're welcome! Whenever you're ready, just send the photos and let me know if you'd like a replacement or refund for order 48213, and we'll take it from there.
[258 tokens in, 125 out, 72 of them thinking]



0

The single question came back in one sentence with its usage, 28 tokens in and 59 out. In the chat, the second reply named order 48213, which only the first message mentioned, and its input had grown to 193 tokens because the first exchange traveled with it. By the third turn the input was 258 tokens. The client keeps the conversation the same way the earlier cells did, as a list of messages sent in full on every call.

## What the notebook showed

- A request needs a model, `max_tokens` and `messages`, and the response's text sits in a list of content blocks.
- The JSON the SDK sends is exactly those fields, so the recording shows the wire format.
- The model knows only what the request carries. With two earlier turns it answered with the order number, and without them it couldn't.
- Two worked examples turned paragraph-long classifications into one-word labels of 4 or 5 tokens.
- The system prompt took the same question from 644 output tokens to 58, or to JSON that parsed.
- A conversation that ends with an assistant turn gets a 400 error on Claude Sonnet 5.5, as of November 2026.
- Replay answers a request only if it matches the recorded one exactly, so an edited cell fails with a message that says what to do.

---
<div style="text-align:center;color:#666f8e;font-size:12px;margin-top:10px">Building AI with AI: A Visual Handbook · © 2026 John Fisher · text CC BY 4.0, code MIT</div>